# 04 - Experimentos CNN

En esta notebook se prueban variaciones de hiperparámetros para CNNs:

- Batch Normalization;
- Dropout;
- Weight Decay;
- Learning Rate;
- Data Augmentation;
- capacidad del modelo.

Primero se realiza un screening rápido usando un solo fold. Luego se corren en validación cruzada completa los mejores candidatos.

El test fijo no se usa en esta notebook.

In [1]:
from pathlib import Path
import os
import sys
import json

import pandas as pd

REPO_ROOT = Path.cwd()

if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)

src_path = REPO_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from cnn_training import run_cross_validation, save_experiment_outputs

print("Repo root:", REPO_ROOT)

Repo root: c:\Users\tomas\OneDrive\Documentos\MATERIAS ITBA\ELECTIVAS - CUATRIMESTRE X\MACHINE LEARNING y REDES NEURONALES EN BIOINGENIERÍA\skin-dataset-classification


In [2]:
base_config = {
    "mlflow_experiment": "TP_skin_CNN",

    "model": "AlexNetSmall",
    "image_size": 128,
    "batch_size": 32,
    "epochs": 15,
    "lr": 3e-4,
    "optimizer": "adamw",

    "dropout": 0.3,
    "batch_norm": True,
    "weight_decay": 1e-4,
    "augmentation": "light",

    "early_stopping": True,
    "patience": 5,

    "tensorboard": True,
    "mlflow": True,
    "log_pytorch_model": False,
    "histogram_every": 5,

    "seed": 42,
}

In [3]:
screening_configs = []

def make_config(name, **updates):
    cfg = base_config.copy()
    cfg["experiment"] = name
    cfg.update(updates)
    return cfg

screening_configs.append(
    make_config(
        "CNN_SCREEN_1_no_batchnorm",
        batch_norm=False,
    )
)

screening_configs.append(
    make_config(
        "CNN_SCREEN_2_no_dropout",
        dropout=0.0,
    )
)

screening_configs.append(
    make_config(
        "CNN_SCREEN_3_dropout05",
        dropout=0.5,
    )
)

screening_configs.append(
    make_config(
        "CNN_SCREEN_4_aug_minimal",
        augmentation="minimal",
    )
)

screening_configs.append(
    make_config(
        "CNN_SCREEN_5_aug_medium",
        augmentation="medium",
    )
)

screening_configs.append(
    make_config(
        "CNN_SCREEN_6_weight_decay_1e3",
        weight_decay=1e-3,
    )
)

screening_configs.append(
    make_config(
        "CNN_SCREEN_7_lr_1e3",
        lr=1e-3,
    )
)

screening_configs.append(
    make_config(
        "CNN_SCREEN_8_alexnetmedium",
        model="AlexNetMedium",
    )
)

len(screening_configs), [cfg["experiment"] for cfg in screening_configs]

(8,
 ['CNN_SCREEN_1_no_batchnorm',
  'CNN_SCREEN_2_no_dropout',
  'CNN_SCREEN_3_dropout05',
  'CNN_SCREEN_4_aug_minimal',
  'CNN_SCREEN_5_aug_medium',
  'CNN_SCREEN_6_weight_decay_1e3',
  'CNN_SCREEN_7_lr_1e3',
  'CNN_SCREEN_8_alexnetmedium'])

In [4]:
screening_summaries = []

for cfg in screening_configs:
    print("\n" + "=" * 80)
    print("Corriendo:", cfg["experiment"])
    print("=" * 80)

    output = run_cross_validation(
        config=cfg,
        split_csv="data/splits/final_split_5fold.csv",
        folds_to_run=[0],
    )

    output_prefix = cfg["experiment"].lower()
    save_experiment_outputs(
        cv_output=output,
        output_prefix=output_prefix,
    )

    screening_summaries.append(output["summary"])

screening_df = pd.DataFrame(screening_summaries)
screening_df = screening_df.sort_values("val_accuracy_mean", ascending=False)

display(screening_df[[
    "experiment",
    "model",
    "dropout",
    "batch_norm",
    "weight_decay",
    "lr",
    "augmentation",
    "val_accuracy_mean",
    "val_macro_f1_mean",
    "val_balanced_accuracy_mean",
]])


Corriendo: CNN_SCREEN_1_no_batchnorm
Device usado: cpu
Folds a correr: [0]
Clases: {0: 'Actinic keratosis', 1: 'Atopic Dermatitis', 2: 'Benign keratosis', 3: 'Dermatofibroma', 4: 'Melanocytic nevus', 5: 'Melanoma', 6: 'Squamous cell carcinoma', 7: 'Tinea Ringworm Candidiasis', 8: 'Vascular lesion'}
Parámetros entrenables: 2,734,793
[CNN_SCREEN_1_no_batchnorm] fold 0 | epoch 01/15 | train_acc=0.101 | val_acc=0.201 | val_f1=0.103
[CNN_SCREEN_1_no_batchnorm] fold 0 | epoch 02/15 | train_acc=0.216 | val_acc=0.292 | val_f1=0.202
[CNN_SCREEN_1_no_batchnorm] fold 0 | epoch 03/15 | train_acc=0.277 | val_acc=0.285 | val_f1=0.200
[CNN_SCREEN_1_no_batchnorm] fold 0 | epoch 04/15 | train_acc=0.311 | val_acc=0.271 | val_f1=0.180
[CNN_SCREEN_1_no_batchnorm] fold 0 | epoch 05/15 | train_acc=0.323 | val_acc=0.299 | val_f1=0.221
[CNN_SCREEN_1_no_batchnorm] fold 0 | epoch 06/15 | train_acc=0.342 | val_acc=0.361 | val_f1=0.296
[CNN_SCREEN_1_no_batchnorm] fold 0 | epoch 07/15 | train_acc=0.340 | val_acc=

,experiment,model,dropout,batch_norm,weight_decay,lr,augmentation,val_accuracy_mean,val_macro_f1_mean,val_balanced_accuracy_mean
3,CNN_SCREEN_4_aug_minimal,AlexNetSmall,0.3,True,0.0001,0.0003,minimal,0.666667,0.639821,0.651515
5,CNN_SCREEN_6_weight_decay_1e3,AlexNetSmall,0.3,True,0.0010,0.0003,light,0.638889,0.595755,0.616713
2,CNN_SCREEN_3_dropout05,AlexNetSmall,0.5,True,0.0001,0.0003,light,0.618056,0.570922,0.592904
1,CNN_SCREEN_2_no_dropout,AlexNetSmall,0.0,True,0.0001,0.0003,light,0.618056,0.581046,0.598633
7,CNN_SCREEN_8_alexnetmedium,AlexNetMedium,0.3,True,0.0001,0.0003,light,0.618056,0.587294,0.597233
4,CNN_SCREEN_5_aug_medium,AlexNetSmall,0.3,True,0.0001,0.0003,medium,0.618056,0.585323,0.597869
6,CNN_SCREEN_7_lr_1e3,AlexNetSmall,0.3,True,0.0001,0.0010,light,0.583333,0.537774,0.573720
0,CNN_SCREEN_1_no_batchnorm,AlexNetSmall,0.3,False,0.0001,0.0003,light,0.486111,0.444009,0.467320


In [5]:
screening_summary_path = Path("experiments/cnn_screening_summary.csv")
screening_df.to_csv(screening_summary_path, index=False)
print("Guardado:", screening_summary_path)

Guardado: experiments\cnn_screening_summary.csv


In [6]:
top2_names = screening_df.head(2)["experiment"].tolist()
top2_names

top2_screening_configs = [
    cfg for cfg in screening_configs
    if cfg["experiment"] in top2_names
]

top2_screening_configs

[{'mlflow_experiment': 'TP_skin_CNN',
  'model': 'AlexNetSmall',
  'image_size': 128,
  'batch_size': 32,
  'epochs': 15,
  'lr': 0.0003,
  'optimizer': 'adamw',
  'dropout': 0.3,
  'batch_norm': True,
  'weight_decay': 0.0001,
  'augmentation': 'minimal',
  'early_stopping': True,
  'patience': 5,
  'tensorboard': True,
  'mlflow': True,
  'log_pytorch_model': False,
  'histogram_every': 5,
  'seed': 42,
  'experiment': 'CNN_SCREEN_4_aug_minimal'},
 {'mlflow_experiment': 'TP_skin_CNN',
  'model': 'AlexNetSmall',
  'image_size': 128,
  'batch_size': 32,
  'epochs': 15,
  'lr': 0.0003,
  'optimizer': 'adamw',
  'dropout': 0.3,
  'batch_norm': True,
  'weight_decay': 0.001,
  'augmentation': 'light',
  'early_stopping': True,
  'patience': 5,
  'tensorboard': True,
  'mlflow': True,
  'log_pytorch_model': False,
  'histogram_every': 5,
  'seed': 42,
  'experiment': 'CNN_SCREEN_6_weight_decay_1e3'}]

In [7]:
full_summaries = []

for cfg in top2_screening_configs:
    full_cfg = cfg.copy()

    full_cfg["experiment"] = cfg["experiment"].replace("CNN_SCREEN_", "CNN_FULL_")
    full_cfg["epochs"] = 25
    full_cfg["patience"] = 6
    full_cfg["log_pytorch_model"] = True

    print("\n" + "=" * 80)
    print("Corriendo FULL CV:", full_cfg["experiment"])
    print("=" * 80)

    output = run_cross_validation(
        config=full_cfg,
        split_csv="data/splits/final_split_5fold.csv",
        folds_to_run=None,
    )

    output_prefix = full_cfg["experiment"].lower()
    save_experiment_outputs(
        cv_output=output,
        output_prefix=output_prefix,
    )

    full_summaries.append(output["summary"])

full_df = pd.DataFrame(full_summaries)
full_df = full_df.sort_values("val_accuracy_mean", ascending=False)

display(full_df[[
    "experiment",
    "model",
    "dropout",
    "batch_norm",
    "weight_decay",
    "lr",
    "augmentation",
    "val_accuracy_mean",
    "val_accuracy_std",
    "val_macro_f1_mean",
    "val_balanced_accuracy_mean",
]])


Corriendo FULL CV: CNN_FULL_4_aug_minimal
Device usado: cpu
Folds a correr: [0, 1, 2, 3, 4]
Clases: {0: 'Actinic keratosis', 1: 'Atopic Dermatitis', 2: 'Benign keratosis', 3: 'Dermatofibroma', 4: 'Melanocytic nevus', 5: 'Melanoma', 6: 'Squamous cell carcinoma', 7: 'Tinea Ringworm Candidiasis', 8: 'Vascular lesion'}
Parámetros entrenables: 2,736,009
[CNN_FULL_4_aug_minimal] fold 0 | epoch 01/25 | train_acc=0.243 | val_acc=0.368 | val_f1=0.287
[CNN_FULL_4_aug_minimal] fold 0 | epoch 02/25 | train_acc=0.373 | val_acc=0.479 | val_f1=0.426
[CNN_FULL_4_aug_minimal] fold 0 | epoch 03/25 | train_acc=0.442 | val_acc=0.549 | val_f1=0.488
[CNN_FULL_4_aug_minimal] fold 0 | epoch 04/25 | train_acc=0.517 | val_acc=0.604 | val_f1=0.572
[CNN_FULL_4_aug_minimal] fold 0 | epoch 05/25 | train_acc=0.522 | val_acc=0.583 | val_f1=0.535
[CNN_FULL_4_aug_minimal] fold 0 | epoch 06/25 | train_acc=0.597 | val_acc=0.597 | val_f1=0.545
[CNN_FULL_4_aug_minimal] fold 0 | epoch 07/25 | train_acc=0.609 | val_acc=0.62

2026/06/18 10:57:27 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Archivos guardados:
- experiments\cnn_full_4_aug_minimal_fold_results.csv
- experiments\cnn_full_4_aug_minimal_history.csv
- experiments\cnn_full_4_aug_minimal_summary.json
- results\training_curves\cnn_full_4_aug_minimal_loss.png
- results\training_curves\cnn_full_4_aug_minimal_accuracy.png
- results\confusion_matrices\cnn_full_4_aug_minimal.png
- results\classification_reports\cnn_full_4_aug_minimal_classification_report.txt

Corriendo FULL CV: CNN_FULL_6_weight_decay_1e3
Device usado: cpu
Folds a correr: [0, 1, 2, 3, 4]
Clases: {0: 'Actinic keratosis', 1: 'Atopic Dermatitis', 2: 'Benign keratosis', 3: 'Dermatofibroma', 4: 'Melanocytic nevus', 5: 'Melanoma', 6: 'Squamous cell carcinoma', 7: 'Tinea Ringworm Candidiasis', 8: 'Vascular lesion'}
Parámetros entrenables: 2,736,009
[CNN_FULL_6_weight_decay_1e3] fold 0 | epoch 01/25 | train_acc=0.215 | val_acc=0.347 | val_f1=0.267
[CNN_FULL_6_weight_decay_1e3] fold 0 | epoch 02/25 | train_acc=0.321 | val_acc=0.438 | val_f1=0.359
[CNN_FULL_6_

2026/06/18 12:22:19 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Archivos guardados:
- experiments\cnn_full_6_weight_decay_1e3_fold_results.csv
- experiments\cnn_full_6_weight_decay_1e3_history.csv
- experiments\cnn_full_6_weight_decay_1e3_summary.json
- results\training_curves\cnn_full_6_weight_decay_1e3_loss.png
- results\training_curves\cnn_full_6_weight_decay_1e3_accuracy.png
- results\confusion_matrices\cnn_full_6_weight_decay_1e3.png
- results\classification_reports\cnn_full_6_weight_decay_1e3_classification_report.txt


,experiment,model,dropout,batch_norm,weight_decay,lr,augmentation,val_accuracy_mean,val_accuracy_std,val_macro_f1_mean,val_balanced_accuracy_mean
0,CNN_FULL_4_aug_minimal,AlexNetSmall,0.3,True,0.0001,0.0003,minimal,0.679157,0.020665,0.673943,0.673236
1,CNN_FULL_6_weight_decay_1e3,AlexNetSmall,0.3,True,0.0010,0.0003,light,0.637345,0.014052,0.619341,0.627523


In [8]:
baseline_summary_path = Path("experiments/cnn_0_alexnetsmall_bn_dropout_wd_lightaug_summary.json")

with open(baseline_summary_path, "r", encoding="utf-8") as f:
    baseline_summary = json.load(f)

baseline_row = {
    "experiment": baseline_summary["experiment"],
    "model": baseline_summary["model"],
    "dropout": baseline_summary["dropout"],
    "batch_norm": baseline_summary["batch_norm"],
    "weight_decay": baseline_summary["weight_decay"],
    "lr": baseline_summary["lr"],
    "augmentation": baseline_summary["augmentation"],
    "val_accuracy_mean": baseline_summary["val_accuracy_mean"],
    "val_accuracy_std": baseline_summary["val_accuracy_std"],
    "val_macro_f1_mean": baseline_summary["val_macro_f1_mean"],
    "val_balanced_accuracy_mean": baseline_summary["val_balanced_accuracy_mean"],
}

comparison_cnn_df = pd.concat(
    [
        pd.DataFrame([baseline_row]),
        full_df[[
            "experiment",
            "model",
            "dropout",
            "batch_norm",
            "weight_decay",
            "lr",
            "augmentation",
            "val_accuracy_mean",
            "val_accuracy_std",
            "val_macro_f1_mean",
            "val_balanced_accuracy_mean",
        ]],
    ],
    ignore_index=True,
)

comparison_cnn_df = comparison_cnn_df.sort_values("val_accuracy_mean", ascending=False)

display(comparison_cnn_df)

,experiment,model,dropout,batch_norm,weight_decay,lr,augmentation,val_accuracy_mean,val_accuracy_std,val_macro_f1_mean,val_balanced_accuracy_mean
1,CNN_FULL_4_aug_minimal,AlexNetSmall,0.3,True,0.0001,0.0003,minimal,0.679157,0.020665,0.673943,0.673236
0,CNN_0_alexnetsmall_bn_dropout_wd_lightaug,AlexNetSmall,0.3,True,0.0001,0.0003,light,0.644318,0.021674,0.632304,0.639593
2,CNN_FULL_6_weight_decay_1e3,AlexNetSmall,0.3,True,0.0010,0.0003,light,0.637345,0.014052,0.619341,0.627523


In [9]:
comparison_cnn_path = Path("experiments/cnn_full_comparison.csv")
comparison_cnn_df.to_csv(comparison_cnn_path, index=False)
print("Guardado:", comparison_cnn_path)

Guardado: experiments\cnn_full_comparison.csv
